**ETL**

In [4]:
import os
from pathlib import Path

import pandas as pd

In [2]:
import os
import pandas as pd

**Extraindo e juntando os dados**

In [5]:
# Caminho dos arquivos com os dados
arquivos_path = Path("..") / "Datas"
arquivos_name_list = sorted([arquivo.name for arquivo in arquivos_path.glob("*.csv")])

# Carrega os arquivos como DataFrames do pandas
# Mantemos tudo como texto na leitura para preservar CNPJs e padronizar a limpeza depois.
tdf_dict = {}
for arquivo_name in arquivos_name_list:
    try:
        path = arquivos_path / arquivo_name
        tdf = pd.read_csv(path, sep=";", encoding="cp1252", dtype=str)
        tdf["arquivo_origem"] = arquivo_name
        tdf_dict[arquivo_name] = tdf
    except Exception as e:
        print(f"Erro ao ler o arquivo '{arquivo_name}': {e}")

In [6]:
# Junta os dados em um unico DataFrame
df = pd.concat(tdf_dict.values(), ignore_index=True)

# Padroniza nomes de colunas e remove espacos extras

# Remove espacos extras de todas as colunas de texto
# e converte campos de data e valor para tipos adequados.
df.columns = df.columns.str.strip()
text_columns = [
    coluna for coluna in df.columns
    if pd.api.types.is_string_dtype(df[coluna]) or df[coluna].dtype == object
]
for coluna in text_columns:
    df[coluna] = df[coluna].str.strip()

df["DatGeracaoConjuntoDados"] = pd.to_datetime(df["DatGeracaoConjuntoDados"], errors="coerce")
df["DatCompetencia"] = pd.to_datetime(df["DatCompetencia"], errors="coerce")
df["VlrMercado"] = pd.to_numeric(
    df["VlrMercado"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce",
)

# CNPJ deve ser tratado como identificador textual
for coluna in ["NumCNPJAgenteDistribuidora", "NumCNPJAgenteAcessante"]:
    df[coluna] = (
        df[coluna]
        .str.replace(r"\D", "", regex=True)
        .replace({"": pd.NA})
    )

**Analizando e transformando dados**

Foram utilizados os dados de 2024 a 2026. O arquivo de 2026 ainda pode estar em atualização, então vale conferir o volume antes de usar comparações finais.

1. Informações gerais dos dados

In [5]:
df.head(5)

,DatGeracaoConjuntoDados,NumCNPJAgenteDistribuidora,SigAgenteDistribuidora,NomAgenteDistribuidora,NomTipoMercado,DscModalidadeTarifaria,DscSubGrupoTarifario,DscClasseConsumoMercado,DscSubClasseConsumidor,DscDetalheConsumidor,IdeAgenteAcessante,NumCNPJAgenteAcessante,NomAgenteAcessante,DscPostoTarifario,DscOpcaoEnergia,DscDetalheMercado,DatCompetencia,VlrMercado
0,2026-05-15,4065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Sistema Isolado - Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 04,Não se aplica,NaN,3.712167e+10,Não se aplica,Não se aplica,CATIVO,Energia TE (kWh),2024-01-01,"583151,000000"
1,2026-05-15,4065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 01,Não se aplica,NaN,3.712167e+10,Não se aplica,Não se aplica,CATIVO,PIS/PASEP (R$),2024-01-01,"5431,290000"
2,2026-05-15,4065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Azul,A4,Serviço público,"Água, esgoto e saneamento",Não se aplica,NaN,3.712167e+10,Não se aplica,Ponta,CATIVO,Receita Demanda (R$),2024-01-01,"28137,260000"
3,2026-05-15,4065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Sistema de Compensação GD II,Branca,B1,Residencial,Residencial,Não se aplica,NaN,3.712167e+10,Não se aplica,Fora ponta,CATIVO,Receita energia compensada (R$),2024-01-01,"235,940000"
4,2026-05-15,4065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 04,Não se aplica,NaN,3.712167e+10,Não se aplica,Não se aplica,CATIVO,Receita Energia (R$),2024-01-01,"2061470,440000"


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3029329 entries, 0 to 3029328
Data columns (total 18 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   DatGeracaoConjuntoDados     str    
 1   NumCNPJAgenteDistribuidora  int64  
 2   SigAgenteDistribuidora      str    
 3   NomAgenteDistribuidora      str    
 4   NomTipoMercado              str    
 5   DscModalidadeTarifaria      str    
 6   DscSubGrupoTarifario        str    
 7   DscClasseConsumoMercado     str    
 8   DscSubClasseConsumidor      str    
 9   DscDetalheConsumidor        str    
 10  IdeAgenteAcessante          float64
 11  NumCNPJAgenteAcessante      float64
 12  NomAgenteAcessante          str    
 13  DscPostoTarifario           str    
 14  DscOpcaoEnergia             str    
 15  DscDetalheMercado           str    
 16  DatCompetencia              str    
 17  VlrMercado                  str    
dtypes: float64(2), int64(1), str(15)
memory usage: 416.0 MB


Inicialmente é possivel observar que o conjunto de dados tem 3.780.041 linhas e 18 colunas; com 15 colunas sendo do tipo string, 2 sendo float64 e 1 sendo int64.

Segue uma descrição sobre cada coluna e o que ela representa:

01. DatGeracaoConjuntoDados:
02. NumCNPJAgenteDistribuidora: CNPJ da agencia distribuidora de energia.
03. SigAgenteDistribuidora: Sigla utilizada para idantificar a agencia distribuidora de energia.
04. NomAgenteDistribuidora: Nome completo da agencia distribuidora de energia.
05. NomTipoMercado: Identifica qual o tipo de mercado de energia elétrica do registro.
06. DscModalidadeTarifaria: Indica o tipo de cobrança aplicada ao fornecimento de energia elétrica daquele registro.
07. DscSubGrupoTarifario: Indica o subgrupo tarifário ao qual a modalidade tarifaria faz parte.
08. DscClasseConsumoMercado: 
09. DscSubClasseConsumidor:
10. DscDetalheConsumidor:
11. IdeAgenteAcessante:
12. NumCNPJAgenteAcessante:
13. NomAgenteAcessante:
14. DscPostoTarifario: indica o período do dia ao qual o consumo ou a demanda de energia está associado.
15. DscOpcaoEnergia:
16. DscDetalheMercado: Indica o que o registro está medindo no VlrMercado.
17. DatCompetencia:
18. VlrMercado:

In [ ]:
postos_tarifarios = df["DscPostoTarifario"].dropna().unique()
print(postos_tarifarios)

<StringArray>
['Não se aplica', 'Ponta', 'Fora ponta', 'Intermediário', 'Fora Ponta']
Length: 5, dtype: str


2. Tipos dos dados

Ao observar os tipos é possivel notar que á 2 colunas com CNPJ (NumCNPJAgenteDistribuidora e NumCNPJAgenteAcessante) estão como int64 e float64, o que é ruim já que ele deve ser interpretado como um valor identificador e não como um numero.

In [13]:
df[["NumCNPJAgenteDistribuidora", "NumCNPJAgenteAcessante"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 3029329 entries, 0 to 3029328
Data columns (total 2 columns):
 #   Column                      Dtype
---  ------                      -----
 0   NumCNPJAgenteDistribuidora  str  
 1   NumCNPJAgenteAcessante      str  
dtypes: str(2)
memory usage: 46.2 MB


**3. Camada Silver - limpeza e padronização**

Nesta etapa tratamos valores nulos, removemos duplicidades, padronizamos textos e criamos colunas auxiliares para análise.

In [7]:
silver_df = df.copy()

# Registra metricas antes de alterar a base.
linhas_duplicadas_exatas = int(silver_df.duplicated().sum())
silver_df_sem_dup = silver_df.drop_duplicates().reset_index(drop=True)
linhas_sem_competencia_ou_valor = int(silver_df_sem_dup[["DatCompetencia", "VlrMercado"]].isna().any(axis=1).sum())

# Remove duplicidades exatas.
silver_df = silver_df_sem_dup.copy()

# Normaliza textos para evitar inconsistencias de capitalização/espacos.
textual_columns = [
    coluna for coluna in silver_df.columns
    if pd.api.types.is_string_dtype(silver_df[coluna])
]
for coluna in textual_columns:
    silver_df[coluna] = (
        silver_df[coluna]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Remove registros sem valor de mercado e sem data de competencia, pois nao sao uteis para agregacao.
silver_df = silver_df.dropna(subset=["DatCompetencia", "VlrMercado"])

# Colunas auxiliares para analise temporal.
silver_df["competencia_ano"] = silver_df["DatCompetencia"].dt.year
silver_df["competencia_mes"] = silver_df["DatCompetencia"].dt.to_period("M").astype(str)
silver_df["competencia_trimestre"] = silver_df["DatCompetencia"].dt.to_period("Q").astype(str)

# Padroniza colunas de identificacao como texto sem simbolos.
for coluna in ["NumCNPJAgenteDistribuidora", "NumCNPJAgenteAcessante"]:
    silver_df[coluna] = silver_df[coluna].astype("string")

# Preenche nulos dos atributos de acessante para evitar problemas em filtros de BI.
silver_df["IdeAgenteAcessante"] = silver_df["IdeAgenteAcessante"].fillna("Nao Informado")
silver_df["NomAgenteAcessante"] = silver_df["NomAgenteAcessante"].fillna("Nao Informado")
silver_df["NumCNPJAgenteAcessante"] = silver_df["NumCNPJAgenteAcessante"].fillna("00000000000000")

silver_df.head()

,DatGeracaoConjuntoDados,NumCNPJAgenteDistribuidora,SigAgenteDistribuidora,NomAgenteDistribuidora,NomTipoMercado,DscModalidadeTarifaria,DscSubGrupoTarifario,DscClasseConsumoMercado,DscSubClasseConsumidor,DscDetalheConsumidor,...,NomAgenteAcessante,DscPostoTarifario,DscOpcaoEnergia,DscDetalheMercado,DatCompetencia,VlrMercado,arquivo_origem,competencia_ano,competencia_mes,competencia_trimestre
0,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Sistema Isolado - Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 04,Não se aplica,...,Não se aplica,Não se aplica,CATIVO,Energia TE (kWh),2024-01-01,583151.00,samp-2024.csv,2024,2024-01,2024Q1
1,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 01,Não se aplica,...,Não se aplica,Não se aplica,CATIVO,PIS/PASEP (R$),2024-01-01,5431.29,samp-2024.csv,2024,2024-01,2024Q1
2,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Azul,A4,Serviço público,"Água, esgoto e saneamento",Não se aplica,...,Não se aplica,Ponta,CATIVO,Receita Demanda (R$),2024-01-01,28137.26,samp-2024.csv,2024,2024-01,2024Q1
3,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Sistema de Compensação GD II,Branca,B1,Residencial,Residencial,Não se aplica,...,Não se aplica,Fora ponta,CATIVO,Receita energia compensada (R$),2024-01-01,235.94,samp-2024.csv,2024,2024-01,2024Q1
4,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 04,Não se aplica,...,Não se aplica,Não se aplica,CATIVO,Receita Energia (R$),2024-01-01,2061470.44,samp-2024.csv,2024,2024-01,2024Q1


In [ ]:
# Indicadores de qualidade da camada Silver.
resumo_qualidade = pd.DataFrame({
    "metric": [
        "linhas_bronze",
        "linhas_silver",
        "duplicatas_removidas",
        "linhas_sem_competencia_ou_valor",
        "linhas_descartadas_por_nulos",
        "datas_competencia_invalidas",
        "valores_nulos_vlrmercado",
    ],
    "value": [
        len(df),
        len(silver_df),
        linhas_duplicadas_exatas,
        linhas_sem_competencia_ou_valor,
        linhas_sem_competencia_ou_valor,
        int(df["DatCompetencia"].isna().sum()),
        int(df["VlrMercado"].isna().sum()),
    ],
})

resumo_qualidade

**4. Camada Gold - esquema estrela**

Nesta etapa transformamos a camada Silver em dimensões e fato para consultas analíticas mais consistentes.

In [8]:
processed_root = arquivos_path / "processed"
bronze_dir = processed_root / "bronze"
silver_dir = processed_root / "silver"
gold_dir = processed_root / "gold"

bronze_dir.mkdir(parents=True, exist_ok=True)
silver_dir.mkdir(parents=True, exist_ok=True)
gold_dir.mkdir(parents=True, exist_ok=True)

bronze_path = bronze_dir / "bronze_dataset.csv"
silver_path = silver_dir / "silver_dataset.csv"
dim_tempo_path = gold_dir / "dim_tempo.csv"
dim_distribuidora_path = gold_dir / "dim_distribuidora.csv"
dim_acessante_path = gold_dir / "dim_acessante.csv"
dim_mercado_path = gold_dir / "dim_mercado.csv"
fato_energia_path = gold_dir / "fato_energia.csv"

# Dimensão tempo
dim_tempo = (
    silver_df[["DatCompetencia"]]
    .drop_duplicates()
    .dropna()
    .sort_values("DatCompetencia")
    .reset_index(drop=True)
    .assign(
        tempo_id=lambda tabela: tabela.index + 1,
        ano=lambda tabela: tabela["DatCompetencia"].dt.year,
        mes=lambda tabela: tabela["DatCompetencia"].dt.month,
        nome_mes=lambda tabela: tabela["DatCompetencia"].dt.strftime("%B"),
        trimestre=lambda tabela: tabela["DatCompetencia"].dt.quarter,
        semestre=lambda tabela: ((tabela["DatCompetencia"].dt.month - 1) // 6) + 1,
    )
    [["tempo_id", "DatCompetencia", "ano", "mes", "nome_mes", "trimestre", "semestre"]]
    .rename(columns={"DatCompetencia": "data_competencia"})
)

# Dimensão distribuidora

dim_distribuidora = (
    silver_df[["NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora"]]
    .drop_duplicates()
    .sort_values(["SigAgenteDistribuidora", "NomAgenteDistribuidora"])
    .reset_index(drop=True)
    .assign(distribuidora_id=lambda tabela: tabela.index + 1)
    [["distribuidora_id", "NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora"]]
)

# Dimensão acessante

dim_acessante = (
    silver_df[["NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante"]]
    .drop_duplicates()
    .sort_values(["NomAgenteAcessante", "NumCNPJAgenteAcessante"])
    .reset_index(drop=True)
    .assign(acessante_id=lambda tabela: tabela.index + 1)
    [["acessante_id", "NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante"]]
)

# Dimensão mercado

dim_mercado = (
    silver_df[[
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscSubGrupoTarifario",
        "DscClasseConsumoMercado",
        "DscSubClasseConsumidor",
        "DscDetalheConsumidor",
        "DscPostoTarifario",
        "DscOpcaoEnergia",
        "DscDetalheMercado",
    ]]
    .drop_duplicates()
    .sort_values([
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscPostoTarifario",
        "DscDetalheMercado",
    ])
    .reset_index(drop=True)
    .assign(mercado_id=lambda tabela: tabela.index + 1)
    [[
        "mercado_id",
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscSubGrupoTarifario",
        "DscClasseConsumoMercado",
        "DscSubClasseConsumidor",
        "DscDetalheConsumidor",
        "DscPostoTarifario",
        "DscOpcaoEnergia",
        "DscDetalheMercado",
    ]]
)

# Tabela fato
fato_energia = silver_df[[
    "DatGeracaoConjuntoDados",
    "DatCompetencia",
    "NumCNPJAgenteDistribuidora",
    "SigAgenteDistribuidora",
    "NomAgenteDistribuidora",
    "NumCNPJAgenteAcessante",
    "IdeAgenteAcessante",
    "NomAgenteAcessante",
    "NomTipoMercado",
    "DscModalidadeTarifaria",
    "DscSubGrupoTarifario",
    "DscClasseConsumoMercado",
    "DscSubClasseConsumidor",
    "DscDetalheConsumidor",
    "DscPostoTarifario",
    "DscOpcaoEnergia",
    "DscDetalheMercado",
    "VlrMercado",
    "arquivo_origem",
]].copy()

fato_energia = fato_energia.merge(
    dim_tempo[["tempo_id", "data_competencia"]],
    left_on="DatCompetencia",
    right_on="data_competencia",
    how="left",
    validate="m:1",
)
fato_energia = fato_energia.merge(
    dim_distribuidora[["distribuidora_id", "NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora"]],
    on=["NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora"],
    how="left",
    validate="m:1",
)
fato_energia = fato_energia.merge(
    dim_acessante[["acessante_id", "NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante"]],
    on=["NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante"],
    how="left",
    validate="m:1",
)
fato_energia = fato_energia.merge(
    dim_mercado[[
        "mercado_id",
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscSubGrupoTarifario",
        "DscClasseConsumoMercado",
        "DscSubClasseConsumidor",
        "DscDetalheConsumidor",
        "DscPostoTarifario",
        "DscOpcaoEnergia",
        "DscDetalheMercado",
    ]],
    on=[
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscSubGrupoTarifario",
        "DscClasseConsumoMercado",
        "DscSubClasseConsumidor",
        "DscDetalheConsumidor",
        "DscPostoTarifario",
        "DscOpcaoEnergia",
        "DscDetalheMercado",
    ],
    how="left",
    validate="m:1",
)

fato_energia = fato_energia.drop(columns=["data_competencia"])
fato_energia = fato_energia.assign(fato_id=lambda tabela: tabela.index + 1)[[
    "fato_id",
    "tempo_id",
    "distribuidora_id",
    "acessante_id",
    "mercado_id",
    "DatGeracaoConjuntoDados",
    "VlrMercado",
    "arquivo_origem",
]]

# Exporta Bronze, Silver e o esquema estrela com CSV separado por virgula.
df.to_csv(bronze_path, index=False, encoding="utf-8-sig")
silver_df.to_csv(silver_path, index=False, encoding="utf-8-sig")
dim_tempo.to_csv(dim_tempo_path, index=False, encoding="utf-8-sig")
dim_distribuidora.to_csv(dim_distribuidora_path, index=False, encoding="utf-8-sig")
dim_acessante.to_csv(dim_acessante_path, index=False, encoding="utf-8-sig")
dim_mercado.to_csv(dim_mercado_path, index=False, encoding="utf-8-sig")
fato_energia.to_csv(fato_energia_path, index=False, encoding="utf-8-sig")

{
    "bronze_path": str(bronze_path),
    "silver_path": str(silver_path),
    "dim_tempo_path": str(dim_tempo_path),
    "dim_distribuidora_path": str(dim_distribuidora_path),
    "dim_acessante_path": str(dim_acessante_path),
    "dim_mercado_path": str(dim_mercado_path),
    "fato_energia_path": str(fato_energia_path),
}

{'bronze_path': '..\\Datas\\processed\\bronze\\bronze_dataset.csv',
 'silver_path': '..\\Datas\\processed\\silver\\silver_dataset.csv',
 'dim_tempo_path': '..\\Datas\\processed\\gold\\dim_tempo.csv',
 'dim_distribuidora_path': '..\\Datas\\processed\\gold\\dim_distribuidora.csv',
 'dim_acessante_path': '..\\Datas\\processed\\gold\\dim_acessante.csv',
 'dim_mercado_path': '..\\Datas\\processed\\gold\\dim_mercado.csv',
 'fato_energia_path': '..\\Datas\\processed\\gold\\fato_energia.csv'}

**5. Validação final do ETL**

Aqui conferimos o volume final das dimensões e da tabela fato geradas no esquema estrela.

In [9]:
print(f"Bronze linhas: {len(df):,}")
print(f"Silver linhas: {len(silver_df):,}")
print(f"Dim tempo linhas: {len(dim_tempo):,}")
print(f"Dim distribuidora linhas: {len(dim_distribuidora):,}")
print(f"Dim acessante linhas: {len(dim_acessante):,}")
print(f"Dim mercado linhas: {len(dim_mercado):,}")
print(f"Fato energia linhas: {len(fato_energia):,}")

display(dim_tempo.head())
display(dim_distribuidora.head())
display(dim_acessante.head())
display(dim_mercado.head())
display(fato_energia.head())

Bronze linhas: 3,029,329
Silver linhas: 3,029,186
Dim tempo linhas: 28
Dim distribuidora linhas: 103
Dim acessante linhas: 1,644
Dim mercado linhas: 12,904
Fato energia linhas: 3,029,186


,tempo_id,data_competencia,ano,mes,nome_mes,trimestre,semestre
0,1,2024-01-01,2024,1,January,1,1
1,2,2024-02-01,2024,2,February,1,1
2,3,2024-03-01,2024,3,March,1,1
3,4,2024-04-01,2024,4,April,2,1
4,5,2024-05-01,2024,5,May,2,1


,distribuidora_id,NumCNPJAgenteDistribuidora,SigAgenteDistribuidora,NomAgenteDistribuidora
0,1,02341467000120,AME,AMAZONAS ENERGIA S.A
1,2,02341470000144,BOA VISTA,Roraima Energia S.A.
2,3,30460297000139,CASTRO-DIS,COOPERATIVA DE DISTRIBUICAO DE ENERGIA ELETRIC...
3,4,05965546000109,CEA,COMPANHIA DE ELETRICIDADE DO AMAPA CEA
4,5,60196987000193,CEDRAP,COOPERATIVA DE ELETRIFICACAO DA REGIAO DO ALTO...


,acessante_id,NumCNPJAgenteAcessante,IdeAgenteAcessante,NomAgenteAcessante
0,1,0,Nao Informado,AFYA
1,2,08220101000180,Nao Informado,ALPEK POLYESTER BRASIL S.A.
2,3,02341467000120,Nao Informado,AMAZONAS ENERGIA S.A
3,4,33050071000158,Nao Informado,AMPLA ENERGIA E SERVICOS S.A.
4,5,0,Nao Informado,APE DIMATEX


,mercado_id,NomTipoMercado,DscModalidadeTarifaria,DscSubGrupoTarifario,DscClasseConsumoMercado,DscSubClasseConsumidor,DscDetalheConsumidor,DscPostoTarifario,DscOpcaoEnergia,DscDetalheMercado
0,1,RTE,Convencional,B1,Residencial,Residencial,Não se aplica,Não se aplica,CATIVO,Receita Energia (R$)
1,2,Refaturamento - Regular,Azul,A4,Serviço público,"Água, esgoto e saneamento",Não se aplica,Fora ponta,CATIVO,Demanda Faturada (kW)
2,3,Refaturamento - Regular,Azul,A3,Industrial,Não se aplica,Não se aplica,Fora ponta,LIVRE,Demanda Faturada (kW)
3,4,Refaturamento - Regular,Azul,A3a,Comercial,Não se aplica,Não se aplica,Fora ponta,CATIVO,Demanda Faturada (kW)
4,5,Refaturamento - Regular,Azul,A2,Serviço público,Tração elétrica,Não se aplica,Fora ponta,CATIVO,Demanda Faturada (kW)


,fato_id,tempo_id,distribuidora_id,acessante_id,mercado_id,DatGeracaoConjuntoDados,VlrMercado,arquivo_origem
0,1,1,70,1009,5364,2026-05-15,583151.00,samp-2024.csv
1,2,1,70,1009,3634,2026-05-15,5431.29,samp-2024.csv
2,3,1,70,1009,3037,2026-05-15,28137.26,samp-2024.csv
3,4,1,70,1009,9528,2026-05-15,235.94,samp-2024.csv
4,5,1,70,1009,3687,2026-05-15,2061470.44,samp-2024.csv
